# **Track Network - Lubrication Switch**

### Data Fetching

In [12]:
import pandas as pd
import psycopg2

def fetch_table_to_dataframe(host_ip, database_name, user, password, table_name, port=5432):
    """
    Connects to PostgreSQL and loads the given table into a Pandas DataFrame.
    """
    try:
        # Connect to PostgreSQL
        connection = psycopg2.connect(
            host=host_ip,
            database=database_name,
            user=user,
            password=password,
            port=port
        )
        print(f"Connected successfully to {database_name} on {host_ip}")

        # Create query
        query = f"SELECT * FROM {table_name};"

        # Load into pandas DataFrame
        df = pd.read_sql_query(query, connection)
        print(f"✅ Fetched {len(df)} rows from '{table_name}'")

        return df

    except Exception as e:
        print(f"❌ Error: {e}")
        return None

    finally:
        if connection:
            connection.close()

# --- Configuration (same as before) ---
HOST_IP = "100.95.110.69"
DATABASE_NAME = "pradigma-extractor"
USER = "postgres"
PASSWORD = "password"
PORT = 5432
TABLE_NAME = "extraction"

df_original = fetch_table_to_dataframe(HOST_IP, DATABASE_NAME, USER, PASSWORD, TABLE_NAME, PORT)

Connected successfully to pradigma-extractor on 100.95.110.69


C:\Users\win 11\AppData\Local\Temp\ipykernel_14180\3393819530.py:23: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, connection)


✅ Fetched 7760 rows from 'extraction'


In [13]:
keywords = ["LubricationSwitch"]

pattern = '|'.join(keywords)

df = df_original.copy(deep=True)
df = df[
    (df['status_id'] == 1) &
    (df['dept_name'] == 'Track-Network') &
    (df['filename'].str.contains(pattern, case=False, na=False))
][['filename', 'workorder_id', 'json_data']]

df.head(5)


,filename,workorder_id,json_data
954,TN_PM_MTH_LubricationSwitch5_4000654646.pdf,4.000655e+09,"{'notification': {'notification_no': 'NA', 'no..."
995,TN_PM_MTH_LubricationSwitch2_4000535202.pdf,4.000535e+09,"{'notification': {'notification_no': 'NA', 'no..."
1004,TN_PM_MTH_LubricationSwitch2_4000545710.pdf,4.000546e+09,"{'notification': {'notification_no': 'NA', 'no..."
1017,TN_PM_MTH_LubricationSwitch2_4000495699.pdf,4.000496e+09,"{'notification': {'notification_no': 'NA', 'no..."
1115,TN_PM_MTH_LubricationSwitch4_4000523590.pdf,4.000524e+09,"{'notification': {'notification_no': 'NA', 'no..."


In [14]:
import re

number_group_map = {
    range(1, 5): 'lubrication_switch_hitachi',
    range(5, 7): 'lubrication_switch_mhedemag',  
    range(7, 8): 'lubrication_switch',  
    range(8, 9): 'lubrication_switch_srb',
}

def get_mech_target(filename):
    match = re.search(r"lubricationswitch(\d)", filename, re.IGNORECASE)
    if not match:
        return None
    num = int(match.group(1))
    for key_range, target in number_group_map.items():
        if num in key_range:
            return target
    return None

def rename_json(row):
    data = row['json_data']
    if not isinstance(data, dict):
        return data
    
    target = get_mech_target(row['filename'])
    if target:
        return {target if k.startswith('lubrication_switch') else k: v for k, v in data.items()}

df['json_data'] = df.apply(rename_json, axis=1)

df


,filename,workorder_id,json_data
954,TN_PM_MTH_LubricationSwitch5_4000654646.pdf,4.000655e+09,"{'notification': {'notification_no': 'NA', 'no..."
995,TN_PM_MTH_LubricationSwitch2_4000535202.pdf,4.000535e+09,"{'notification': {'notification_no': 'NA', 'no..."
1004,TN_PM_MTH_LubricationSwitch2_4000545710.pdf,4.000546e+09,"{'notification': {'notification_no': 'NA', 'no..."
1017,TN_PM_MTH_LubricationSwitch2_4000495699.pdf,4.000496e+09,"{'notification': {'notification_no': 'NA', 'no..."
1115,TN_PM_MTH_LubricationSwitch4_4000523590.pdf,4.000524e+09,"{'notification': {'notification_no': 'NA', 'no..."
...,...,...,...
6763,TN_PM_MTH_LubricationSwitch4_4000495721.pdf,4.000496e+09,"{'notification': {'notification_no': 'NA', 'no..."
6808,TN_PM_MTH_LubricationSwitch4_4000512634.pdf,4.000513e+09,"{'notification': {'notification_no': 'NA', 'no..."
6809,TN_PM_MTH_LubricationSwitch4_4000506760.pdf,4.000507e+09,"{'notification': {'notification_no': 'NA', 'no..."
6815,TN_PM_MTH_LubricationSwitch7_4000453432.pdf,4.000453e+09,"{'notification': {'notification_no': 'NA', 'no..."


In [15]:
import pandas as pd

valid_json = df[df['json_data'].apply(lambda x: isinstance(x, dict))]

# initialize dictionary
key_to_workorders = {}

for _, row in valid_json.iterrows():
    workorder = row['workorder_id']
    data = row['json_data']
    
    for key in data.keys():
        key_to_workorders.setdefault(key, []).append(workorder)

for key, wos in key_to_workorders.items():
    unique_wos = sorted({int(x) for x in wos})
    # print(f"{key} ({len(unique_wos)}): {unique_wos}")
    print(f"{key} ({len(unique_wos)})")



notification (356)
work_order (356)
lubrication_switch_mhedemag (89)
lubrication_switch_hitachi (178)
lubrication_switch (45)
lubrication_switch_srb (44)


In [16]:
import os
import re
import json
import numpy as np
import pandas as pd
from collections import Counter

na_like_values = ['NA', 'N/A', 'NULL', 'NONE', 'NAN']
pattern_na = re.compile(r'^\s*(NA|N/A|NULL|NaN)\s*$', re.IGNORECASE)

def is_na_like(val):
    """Detect NA-like values."""
    if isinstance(val, (list, dict, np.ndarray)):
        return False
    try:
        if pd.isna(val):
            return True
    except Exception:
        pass
    val_str = str(val).strip().upper()
    return val_str in na_like_values

def find_na_keys(d):
    """Return keys in dict where value is NA-like."""
    if not isinstance(d, dict):
        return []
    return [k for k, v in d.items() if is_na_like(v)]

def clean_value(val):
    """Recursively clean NA-like values in dict, list, string."""
    if isinstance(val, str):
        return '' if pattern_na.match(val) else val
    elif isinstance(val, dict):
        return {k: clean_value(v) for k, v in val.items()}
    elif isinstance(val, list):
        return [clean_value(v) for v in val]
    else:
        return '' if pd.isna(val) else val

def extract_leaf_keys(d, parent=''):
    """Extract flattened leaf keys from nested dict."""
    keys = []
    if isinstance(d, dict):
        for k, v in d.items():
            full_key = f"{parent}.{k}" if parent else k
            if isinstance(v, dict):
                keys.extend(extract_leaf_keys(v, full_key))
            else:
                keys.append(full_key)
    return keys

def flatten_with_descriptions(subdict):
    """Flatten JSON dict, incorporating 'description' keys as part of flattened column names."""
    flat = {}

    def recurse(d, parent=''):
        if d is None:
            return
        if isinstance(d, str):
            try:
                d = json.loads(d)
            except json.JSONDecodeError:
                return
        if not isinstance(d, dict):
            return

        for k, v in d.items():
            if len(k) == 1 and k.isalpha():
                new_parent = parent
            else:
                new_parent = f"{parent}.{k}" if parent else k

            if isinstance(v, dict):
                desc = v.get('description')
                if desc:
                    desc_key = desc.lower().replace(' ', '_').replace('/', '_').replace('&', 'and')
                    for sub_k, sub_v in v.items():
                        if sub_k != 'description':
                            flat[f"{new_parent}.{desc_key}.{sub_k}"] = sub_v
                else:
                    recurse(v, new_parent)
            else:
                flat[new_parent] = v

    recurse(subdict)
    return flat

### Lubrication Switch (Hitachi)

In [17]:
import re
import json
import numpy as np
import pandas as pd
from collections import Counter

na_like_values = ['NA', 'N/A', 'NULL', 'NONE', 'NAN']
pattern_na = re.compile(r'^\s*(NA|N/A|NULL|NaN)\s*$', re.IGNORECASE)

def is_na_value(value):
    """Check if a value is considered 'NA' based on the pattern."""
    if pd.isna(value) or value is None:
        return True
    if isinstance(value, str):
        return bool(pattern_na.match(value))
    return False

def clean_value(value):
    """Recursively converts 'NA' string values in dicts/lists to np.nan."""
    if isinstance(value, dict):
        return {k: clean_value(v) for k, v in value.items()}
    elif isinstance(value, list):
        return [clean_value(v) for v in value]
    elif isinstance(value, str) and is_na_value(value):
        return np.nan
    else:
        return value

def find_na_keys(d, parent=''):
    """Extracts flattened keys whose values are considered 'NA' (including np.nan)."""
    na_keys = []
    if isinstance(d, dict):
        for k, v in d.items():
            full_key = f"{parent}.{k}" if parent else k
            if isinstance(v, dict):
                na_keys.extend(find_na_keys(v, full_key))
            elif is_na_value(v): # Checks for string 'NA', None, and np.nan
                na_keys.append(full_key)
    return na_keys

def flattened_json(d):
    """
    Flattens a nested dictionary, specifically handling the 'lubrication_switch_hitachi' structure.

    - Flattens the component list (keys 'a' through 'r') into distinct columns (e.g., 'a.component').
    - Flattens other nested dicts (e.g., 'technician', 'gearbox_oil_levels') into distinct columns (e.g., 'technician.date').
    - Handles 'completed?' by creating a unique column name for it.
    """
    flat_data = {}

    def _flatten(data, parent_key=''):
        if isinstance(data, dict):
            for k, v in data.items():
                new_key = f"{parent_key}.{k}" if parent_key else k

                if len(k) == 1 and 'a' <= k <= 'r' and isinstance(v, dict):
                    _flatten(v, new_key)
                
                elif isinstance(v, dict):
                    _flatten(v, new_key)
                else:
                    flat_data[new_key] = v

    _flatten(d)
    return flat_data

###########################################
""" CODE USAGE """ 
##########################################

df_lubswitch_hitachi = df.copy()

df_lubswitch_hitachi['lubrication_switch_hitachi'] = df_lubswitch_hitachi['json_data'].apply(
    lambda x: x.get('lubrication_switch_hitachi') if isinstance(x, dict) else None
)

df_lubswitch_hitachi = df_lubswitch_hitachi[df_lubswitch_hitachi['lubrication_switch_hitachi'].notnull()].copy()

df_lubswitch_hitachi['workorder_id'] = df_lubswitch_hitachi['workorder_id'].apply(lambda x: int(x) if pd.notnull(x) else None)

df_lubswitch_hitachi['lubrication_switch_hitachi'] = df_lubswitch_hitachi['lubrication_switch_hitachi'].apply(clean_value)

df_lubswitch_hitachi['na_keys'] = df_lubswitch_hitachi['lubrication_switch_hitachi'].apply(find_na_keys)
na_counter = Counter(k for keys in df_lubswitch_hitachi['na_keys'] for k in keys)
na_summary = pd.DataFrame(na_counter.items(), columns=['key', 'na_count']).sort_values('na_count', ascending=False)

flattened_rows = [flattened_json(r) for r in df_lubswitch_hitachi['lubrication_switch_hitachi'].fillna({})]
lubswitch_hitachi = pd.DataFrame(flattened_rows) 
lubswitch_hitachi.index = df_lubswitch_hitachi.index
lubswitch_hitachi['workorder_id'] = df_lubswitch_hitachi['workorder_id'].astype('Int64')
lubswitch_hitachi['filename'] = df_lubswitch_hitachi['filename']

for i, col in enumerate(lubswitch_hitachi.columns, start=1):
    print(f"{i:3d}. {col}")
    if col in ['workorder_id', 'filename']:
        continue
    valid_workorders = lubswitch_hitachi.loc[lubswitch_hitachi[col].notna(), 'workorder_id'].unique()
    if len(valid_workorders) > 0:
        workorder_list = ", ".join(map(str, valid_workorders))
        print(f"   Work Orders with data ({len(valid_workorders)}): {workorder_list}")
        print("-" * 80)


  1. a.component
   Work Orders with data (178): 4000535202, 4000545710, 4000495699, 4000523590, 4000643619, 4000619022, 4000570621, 4000557878, 4000606457, 4000479261, 4000490553, 4000501521, 4000484584, 4000490551, 4000700812, 4000452320, 4000457605, 4000467726, 4000667944, 4000680399, 4000479263, 4000442040, 4000700815, 4000660910, 4000654643, 4000588636, 4000648829, 4000596129, 4000636925, 4000630372, 4000673652, 4000680424, 4000686285, 4000648826, 4000636920, 4000630367, 4000601355, 4000667948, 4000660908, 4000574948, 4000545709, 4000539503, 4000545712, 4000517900, 4000528329, 4000574952, 4000457913, 4000473441, 4000574950, 4000654641, 4000619021, 4000545711, 4000557877, 4000539500, 4000528326, 4000523587, 4000512631, 4000506757, 4000484583, 4000462632, 4000441715, 4000501520, 4000512633, 4000490552, 4000625901, 4000473440, 4000612227, 4000462634, 4000601356, 4000588630, 4000447696, 4000473438, 4000495698, 4000557875, 4000570618, 4000588627, 4000619019, 4000606443, 4000686178, 400

### Lubrication Switch (SRB)

In [18]:
import re
import json
import numpy as np
import pandas as pd
from collections import Counter

na_like_values = ['NA', 'N/A', 'NULL', 'NONE', 'NAN']
pattern_na = re.compile(r'^\s*(NA|N/A|NULL|NaN)\s*$', re.IGNORECASE)

def is_na_value(value):
    """Check if a value is considered 'NA' based on the pattern."""
    if pd.isna(value) or value is None:
        return True
    if isinstance(value, str):
        return bool(pattern_na.match(value))
    return False

def clean_value(value):
    """Recursively converts 'NA' string values in dicts/lists to np.nan."""
    if isinstance(value, dict):
        return {k: clean_value(v) for k, v in value.items()}
    elif isinstance(value, list):
        return [clean_value(v) for v in value]
    elif isinstance(value, str) and is_na_value(value):
        return np.nan
    else:
        return value

def find_na_keys(d, parent=''):
    """Extracts flattened keys whose values are considered 'NA' (including np.nan)."""
    na_keys = []
    if isinstance(d, dict):
        for k, v in d.items():
            full_key = f"{parent}.{k}" if parent else k
            if isinstance(v, dict):
                na_keys.extend(find_na_keys(v, full_key))
            elif is_na_value(v): # Checks for string 'NA', None, and np.nan
                na_keys.append(full_key)
    return na_keys

def flattened_json(d):
    """
    Flattens a nested dictionary, specifically handling the 'lubrication_switch_hitachi' structure.

    - Flattens the component list (keys 'a' through 'r') into distinct columns (e.g., 'a.component').
    - Flattens other nested dicts (e.g., 'technician', 'gearbox_oil_levels') into distinct columns (e.g., 'technician.date').
    - Handles 'completed?' by creating a unique column name for it.
    """
    flat_data = {}

    def _flatten(data, parent_key=''):
        if isinstance(data, dict):
            for k, v in data.items():
                new_key = f"{parent_key}.{k}" if parent_key else k

                if len(k) == 1 and 'a' <= k <= 'r' and isinstance(v, dict):
                    _flatten(v, new_key)
                
                elif isinstance(v, dict):
                    _flatten(v, new_key)
                else:
                    flat_data[new_key] = v

    _flatten(d)
    return flat_data


###########################################
""" CODE USAGE """ 
##########################################

df_lubswitch_srb = df.copy()

df_lubswitch_srb['lubrication_switch_srb'] = df_lubswitch_srb['json_data'].apply(
    lambda x: x.get('lubrication_switch_srb') if isinstance(x, dict) else None
)

df_lubswitch_srb = df_lubswitch_srb[df_lubswitch_srb['lubrication_switch_srb'].notnull()].copy()

df_lubswitch_srb['workorder_id'] = df_lubswitch_srb['workorder_id'].apply(lambda x: int(x) if pd.notnull(x) else None)

df_lubswitch_srb['lubrication_switch_srb'] = df_lubswitch_srb['lubrication_switch_srb'].apply(clean_value)

df_lubswitch_srb['na_keys'] = df_lubswitch_srb['lubrication_switch_srb'].apply(find_na_keys)
na_counter = Counter(k for keys in df_lubswitch_srb['na_keys'] for k in keys)
na_summary = pd.DataFrame(na_counter.items(), columns=['key', 'na_count']).sort_values('na_count', ascending=False)

flattened_rows = [flattened_json(r) for r in df_lubswitch_srb['lubrication_switch_srb'].fillna({})]
lubswitch_srb = pd.DataFrame(flattened_rows)  # final DataFrame
lubswitch_srb.index = df_lubswitch_srb.index
lubswitch_srb['workorder_id'] = df_lubswitch_srb['workorder_id'].astype('Int64')
lubswitch_srb['filename'] = df_lubswitch_srb['filename']

for i, col in enumerate(lubswitch_srb.columns, start=1):
    print(f"{i:3d}. {col}")
    if col in ['workorder_id', 'filename']:
        continue
    valid_workorders = lubswitch_srb.loc[lubswitch_srb[col].notna(), 'workorder_id'].unique()
    if len(valid_workorders) > 0:
        workorder_list = ", ".join(map(str, valid_workorders))
        print(f"   Work Orders with data ({len(valid_workorders)}): {workorder_list}")
        print("-" * 80)

  1. a.component
   Work Orders with data (44): 4000667952, 4000660916, 4000442334, 4000654651, 4000447705, 4000648835, 4000630377, 4000453206, 4000457973, 4000479270, 4000484590, 4000462636, 4000473446, 4000467951, 4000517924, 4000506766, 4000528333, 4000545716, 4000535223, 4000570627, 4000564187, 4000557882, 4000588658, 4000606462, 4000490558, 4000495725, 4000625907, 4000619026, 4000601730, 4000612233, 4000638188, 4000643664, 4000673661, 4000680429, 4000693785, 4000686292, 4000596303, 4000583454, 4000574958, 4000551628, 4000539507, 4000523594, 4000512638, 4000501525
--------------------------------------------------------------------------------
  2. a.description
   Work Orders with data (44): 4000667952, 4000660916, 4000442334, 4000654651, 4000447705, 4000648835, 4000630377, 4000453206, 4000457973, 4000479270, 4000484590, 4000462636, 4000473446, 4000467951, 4000517924, 4000506766, 4000528333, 4000545716, 4000535223, 4000570627, 4000564187, 4000557882, 4000588658, 4000606462, 400049

### Lubrication Switch (MHE-DEMAG)

In [19]:
import re
import json
import numpy as np
import pandas as pd
from collections import Counter

na_like_values = ['NA', 'N/A', 'NULL', 'NONE', 'NAN']
pattern_na = re.compile(r'^\s*(NA|N/A|NULL|NaN)\s*$', re.IGNORECASE)

def is_na_value(value):
    """Check if a value is considered 'NA' based on the pattern."""
    
    if pd.isna(value) or value is None:
        return True
    if isinstance(value, str):
        return bool(pattern_na.match(value))
    return False

def clean_value(value):
    """Recursively converts 'NA' string values in dicts/lists to np.nan."""
    if isinstance(value, dict):
        return {k: clean_value(v) for k, v in value.items()}
    elif isinstance(value, list):
        return [clean_value(v) for v in value]
    elif isinstance(value, str) and is_na_value(value):
        return np.nan
    else:
        return value

def find_na_keys(d, parent=''):
    """Extracts flattened keys whose values are considered 'NA' (including np.nan)."""
    na_keys = []
    if isinstance(d, dict):
        for k, v in d.items():
            full_key = f"{parent}.{k}" if parent else k
            if isinstance(v, dict):
                na_keys.extend(find_na_keys(v, full_key))
            elif is_na_value(v): # Checks for string 'NA', None, and np.nan
                na_keys.append(full_key)
    return na_keys

def flattened_json(d):
    """
    Flattens a nested dictionary, specifically handling the 'lubrication_switch_hitachi' structure.

    - Flattens the component list (keys 'a' through 'r') into distinct columns (e.g., 'a.component').
    - Flattens other nested dicts (e.g., 'technician', 'gearbox_oil_levels') into distinct columns (e.g., 'technician.date').
    - Handles 'completed?' by creating a unique column name for it.
    """
    flat_data = {}

    def _flatten(data, parent_key=''):
        if isinstance(data, dict):
            for k, v in data.items():
                new_key = f"{parent_key}.{k}" if parent_key else k

                if len(k) == 1 and 'a' <= k <= 'r' and isinstance(v, dict):
                    _flatten(v, new_key)
                
                elif isinstance(v, dict):
                    _flatten(v, new_key)
                else:
                    flat_data[new_key] = v

    _flatten(d)
    return flat_data

###########################################
""" CODE USAGE """ 
##########################################

df_lubswitch_mhedemag = df.copy()

df_lubswitch_mhedemag['lubrication_switch_mhedemag'] = df_lubswitch_mhedemag['json_data'].apply(
    lambda x: x.get('lubrication_switch_mhedemag') if isinstance(x, dict) else None
)

df_lubswitch_mhedemag = df_lubswitch_mhedemag[df_lubswitch_mhedemag['lubrication_switch_mhedemag'].notnull()].copy()

df_lubswitch_mhedemag['workorder_id'] = df_lubswitch_mhedemag['workorder_id'].apply(lambda x: int(x) if pd.notnull(x) else None)

df_lubswitch_mhedemag['lubrication_switch_mhedemag'] = df_lubswitch_mhedemag['lubrication_switch_mhedemag'].apply(clean_value)

df_lubswitch_mhedemag['na_keys'] = df_lubswitch_mhedemag['lubrication_switch_mhedemag'].apply(find_na_keys)
na_counter = Counter(k for keys in df_lubswitch_mhedemag['na_keys'] for k in keys)
na_summary = pd.DataFrame(na_counter.items(), columns=['key', 'na_count']).sort_values('na_count', ascending=False)

flattened_rows = [flattened_json(r) for r in df_lubswitch_mhedemag['lubrication_switch_mhedemag'].fillna({})]
lubswitch_mhedemag = pd.DataFrame(flattened_rows)  # final DataFrame
lubswitch_mhedemag.index = df_lubswitch_mhedemag.index
lubswitch_mhedemag['workorder_id'] = df_lubswitch_mhedemag['workorder_id'].astype('Int64')
lubswitch_mhedemag['filename'] = df_lubswitch_mhedemag['filename']

###########################################################
""" CUSTOM RULE CHECKS FOR MHE-DEMAG LUBRICATION SWITCH """
###########################################################

if 'e.interval' in lubswitch_mhedemag.columns:
    lubswitch_mhedemag['e.interval'] = "Monthly"

col_completed = 'e.completed?'
if col_completed in lubswitch_mhedemag.columns:
    lubswitch_mhedemag[col_completed] = "not required"


# for i, col in enumerate(lubswitch_mhedemag.columns, start=1):
#     print(f"{i:3d}. {col}")
#     if col in ['workorder_id', 'filename']:
#         continue
#     valid_workorders = lubswitch_mhedemag.loc[lubswitch_mhedemag[col].notna(), 'workorder_id'].unique()
#     if len(valid_workorders) > 0:
#         workorder_list = ", ".join(map(str, valid_workorders))
#         print(f"   Work Orders with data ({len(valid_workorders)}): {workorder_list}")
#         print("-" * 80)


### Mechanical Switch 

In [20]:
import re
import json
import numpy as np
import pandas as pd
from collections import Counter

na_like_values = ['NA', 'N/A', 'NULL', 'NONE', 'NAN']
pattern_na = re.compile(r'^\s*(NA|N/A|NULL|NaN)\s*$', re.IGNORECASE)

def is_na_value(value):
    """Check if a value is considered 'NA' based on the pattern."""
    if pd.isna(value) or value is None:
        return True
    if isinstance(value, str):
        return bool(pattern_na.match(value))
    return False

def clean_value(value):
    """Recursively converts 'NA' string values in dicts/lists to np.nan."""
    if isinstance(value, dict):
        return {k: clean_value(v) for k, v in value.items()}
    elif isinstance(value, list):
        return [clean_value(v) for v in value]
    elif isinstance(value, str) and is_na_value(value):
        return np.nan
    else:
        return value

def find_na_keys(d, parent=''):
    """Extracts flattened keys whose values are considered 'NA' (including np.nan)."""
    na_keys = []
    if isinstance(d, dict):
        for k, v in d.items():
            full_key = f"{parent}.{k}" if parent else k
            if isinstance(v, dict):
                na_keys.extend(find_na_keys(v, full_key))
            elif is_na_value(v): # Checks for string 'NA', None, and np.nan
                na_keys.append(full_key)
    return na_keys

def flattened_json(d):
    """
    Flattens a nested dictionary, specifically handling the 'lubrication_switch_hitachi' structure.

    - Flattens the component list (keys 'a' through 'r') into distinct columns (e.g., 'a.component').
    - Flattens other nested dicts (e.g., 'technician', 'gearbox_oil_levels') into distinct columns (e.g., 'technician.date').
    - Handles 'completed?' by creating a unique column name for it.
    """
    flat_data = {}

    def _flatten(data, parent_key=''):
        if isinstance(data, dict):
            for k, v in data.items():
                new_key = f"{parent_key}.{k}" if parent_key else k

                if len(k) == 1 and 'a' <= k <= 'r' and isinstance(v, dict):
                    _flatten(v, new_key)
                
                elif isinstance(v, dict):
                    _flatten(v, new_key)
                else:
                    flat_data[new_key] = v

    _flatten(d)
    return flat_data

###########################################
""" CODE USAGE """ 
##########################################

df_lubswitch = df.copy()

df_lubswitch['lubrication_switch'] = df_lubswitch['json_data'].apply(
    lambda x: x.get('lubrication_switch') if isinstance(x, dict) else None
)

df_lubswitch = df_lubswitch[df_lubswitch['lubrication_switch'].notnull()].copy()

df_lubswitch['workorder_id'] = df_lubswitch['workorder_id'].apply(lambda x: int(x) if pd.notnull(x) else None)

df_lubswitch['lubrication_switch'] = df_lubswitch['lubrication_switch'].apply(clean_value)

df_lubswitch['na_keys'] = df_lubswitch['lubrication_switch'].apply(find_na_keys)
na_counter = Counter(k for keys in df_lubswitch['na_keys'] for k in keys)
na_summary = pd.DataFrame(na_counter.items(), columns=['key', 'na_count']).sort_values('na_count', ascending=False)

flattened_rows = [flattened_json(r) for r in df_lubswitch['lubrication_switch'].fillna({})]
lubswitch = pd.DataFrame(flattened_rows)  # final DataFrame
lubswitch.index = df_lubswitch.index
lubswitch['workorder_id'] = df_lubswitch['workorder_id'].astype('Int64')
lubswitch['filename'] = df_lubswitch['filename']

for i, col in enumerate(lubswitch.columns, start=1):
    print(f"{i:3d}. {col}")
    if col in ['workorder_id', 'filename']:
        continue
    valid_workorders = lubswitch.loc[lubswitch[col].notna(), 'workorder_id'].unique()
    if len(valid_workorders) > 0:
        workorder_list = ", ".join(map(str, valid_workorders))
        print(f"   Work Orders with data ({len(valid_workorders)}): {workorder_list}")
        print("-" * 80)


  1. a.component
   Work Orders with data (45): 4000447702, 4000441714, 4000473445, 4000654649, 4000448337, 4000463694, 4000484589, 4000517923, 4000596302, 4000680427, 4000686290, 4000693784, 4000643663, 4000660914, 4000667951, 4000673658, 4000630375, 4000638186, 4000625906, 4000619025, 4000612232, 4000606461, 4000601729, 4000588654, 4000583453, 4000574957, 4000570626, 4000564186, 4000557881, 4000551627, 4000528332, 4000545715, 4000539506, 4000535222, 4000523593, 4000506763, 4000512637, 4000501524, 4000495724, 4000479268, 4000490557, 4000648834, 4000643677, 4000453432, 4000458805
--------------------------------------------------------------------------------
  2. a.description
   Work Orders with data (45): 4000447702, 4000441714, 4000473445, 4000654649, 4000448337, 4000463694, 4000484589, 4000517923, 4000596302, 4000680427, 4000686290, 4000693784, 4000643663, 4000660914, 4000667951, 4000673658, 4000630375, 4000638186, 4000625906, 4000619025, 4000612232, 4000606461, 4000601729, 400058

### Output Excel

In [21]:
import os
import pandas as pd
from openpyxl import Workbook

output_path = '../../output/lubrication_switch.xlsx'

os.makedirs(os.path.dirname(output_path), exist_ok=True)

if not os.path.exists(output_path):
    Workbook().save(output_path)

with pd.ExcelWriter(output_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    lubswitch_hitachi.to_excel(writer, index=False, sheet_name='hitachi'),
    lubswitch_srb.to_excel(writer, index=False, sheet_name='srb'),
    lubswitch_mhedemag.to_excel(writer, index=False, sheet_name='mhe-demag'),
    lubswitch.to_excel(writer, index=False, sheet_name='mechanical-switch'),

print(f"✅ Exported successfully to '{output_path}' (replaced existing sheet)")


✅ Exported successfully to '../../output/lubrication_switch.xlsx' (replaced existing sheet)
